# 02. Baseline YOLO: fold-ensemble submission

Uses the 4 fold models of `02_baseline` (attach that notebook's output as input), no training.
Every fold model predicts the test set and the boxes are fused by Weighted Boxes Fusion.

In [ ]:
import subprocess
import sys
from pathlib import Path

# shared code (src/core) from the repo
if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import pandas as pd

from core import N_FOLDS
from core.cv import environment, existing
from core.data import find_data_root
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.yolo import make_yolo_fns

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

In [ ]:
# same prediction settings as 02_baseline
CFG = dict(name="baseline", imgsz=1280, conf=0.001, iou=0.7, max_det=500)

weights = {k: existing(f"{CFG['name']}_fold{k}.pt", WORK) for k in range(N_FOLDS)}
assert all(weights.values()), f"attach the 02_baseline output; missing: {[k for k, w in weights.items() if w is None]}"
weights

In [ ]:
DATA = find_data_root()
test = pd.read_csv(DATA / "test_images.csv")
_, predict_fn = make_yolo_fns(CFG, WORK)
test_dets = fold_ensemble(CFG["name"], predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{CFG['name']}_ensemble.csv").head()